# DAPRO Walkthrough: Metric Estimation

This notebook gives an end-to-end, minimal demo of the metric estimation pipeline in this repository:

1. Load synthetic survival data.
2. Build model-style predictions (conditional hazards + quantiles).
3. Compute Oracle (ground truth) metrics.
4. Run DAPRO to allocate budget under an average budget constraint and simulate trajectory data.
5. Evaluate safety metrics (CJR, RMTTU, Cost per Jailbreak) using IPCW estimators.


## 1) Imports and path setup
Run this notebook from `notebooks/` (default), or any location inside the repo.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Make `src/` importable whether we run from repo root or `notebooks/`.
cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.dataset_utils.data_utils import get_data
from src.safety_evaluation.survival_utils.conditional_pmf_utils import get_conditional_pmf
from src.safety_evaluation.survival_utils.compute_mean_time_given_pmf import compute_quantile_survival_time
from src.safety_evaluation.utils.utils import split_data
from src.safety_evaluation.budget_allocators.DAPRO import DAPRO
from src.train_model.models.utils import SurvivalModelPrediction
from src.safety_evaluation.estimate_metrics import (
    compute_oracle_metric,
    IPCWTrajectorySimulator,
    MetricsEngine,
    CumulativeJailbreakRateMetric,
    RestrictedMeanTimeToUnsafeMetric,
    TotalBudgetUsed,
    ObservedJailbreaks,
    CostPerJailbreakMetric,
    SurvivalQuantilesMetric
)

torch.set_printoptions(precision=3, sci_mode=False)

## 2) Experiment configuration
These defaults are intentionally small and CPU-friendly.

In [ ]:
device = torch.device("cpu")
seed = 7
cal_size = 1000
budget_per_sample = 10.0
tau_prior = 0.56

# LPB code path in this repo uses a dense tau range in log-space.
taus_range = torch.tensor(np.logspace(-3, -0.01, 500), dtype=torch.float32)

# DAPRO controls
projection = "platt"
score = "prob"
n1 = 100

## 3) Load synthetic data and build prediction tensors
For a lightweight demo, we treat synthetic hazards as model predictions.

In [ ]:
(
    p_train, p_cal, p_test,
    x_train, x_cal, x_test,
    y_train, y_cal, y_test,
    t_tilde_train, t_tilde_cal, t_tilde_test,
    e_train, e_cal, e_test,
    b_train, b_cal, b_test,
    n_samples_train, n_samples_cal, n_samples_test,
) = get_data(
    is_real=False,
    device=device,
    dataset_name="synthetic",
    data_setup="default",
    load_x=False,
    seed=seed,
)

# Build a single pool (cal+test) and derive conditional PMFs + quantiles.
probability_est_all = torch.cat([p_cal, p_test], dim=0).float()
t_tilde_all = torch.cat([t_tilde_cal, t_tilde_test], dim=0).long()
conditional_grid_all = get_conditional_pmf(probability_est_all)

# Quantile estimates f(x, tau) used by calibration/allocators.
quantile_est_all = torch.cat([
    compute_quantile_survival_time(
        conditional_grid_all[:, 0].unsqueeze(1),
        quantile=float(tau),
        tail_distribution="geometric",
    )
    for tau in taus_range
], dim=1).float()

max_time = int(probability_est_all.shape[1])
quantile_est_all.shape, conditional_grid_all.shape

## 4) Compute Oracle Metrics
Compute the ground truth metrics on the dataset for evaluation.

In [ ]:
oracle_metrics = compute_oracle_metric(t_tilde_all, max_time=max_time)
print("Oracle CJR:", oracle_metrics['cjr'])
print("Oracle RMTTU:", oracle_metrics['rmttu'])
print("Oracle Quantiles:", oracle_metrics['quantiles'])

## 5) Create calibration/test split and initialize DAPRO
We initialize DAPRO using the calibration set.

In [ ]:
test_size = len(probability_est_all) - cal_size

(
    _x_cal, _x_test,
    t_cal, prob_cal, q_cal,
    t_test, prob_test, q_test,
    cal_idx, test_idx,
) = split_data(
    seed=seed,
    cal_size=cal_size,
    test_size=test_size,
    x_cal_test=None,
    t_tilde_cal_test=t_tilde_all,
    probability_est=probability_est_all,
    quantile_est_cal_test=quantile_est_all,
)

conditional_grid_cal = conditional_grid_all[cal_idx]

test_pred = SurvivalModelPrediction(quantile_est=q_test, probability_est=prob_test)

m_upper_bound = max_time
allocator = DAPRO(
    conditional_grid=conditional_grid_cal,
    budget_per_sample=budget_per_sample,
    taus_range=taus_range,
    tau_prior=tau_prior,
    m_upper_bound=m_upper_bound,
    projection=projection,
    score=score,
    n1=n1,
)

## 6) Simulate Trajectories & Estimate Metrics
Using the DAPRO allocator, we simulate truncated evaluation and use the `MetricsEngine` to compute unbiased estimates.

In [ ]:
# Simulate trajectories on the test set
traj_data = IPCWTrajectorySimulator.simulate(
    allocator=allocator, 
    x=None, 
    model_prediction=test_pred, 
    t_tilde=t_test, 
    max_time=max_time
)

# Initialize the Metrics Engine
engine = MetricsEngine([
    CumulativeJailbreakRateMetric(oracle_cjr=oracle_metrics['cjr']),
    RestrictedMeanTimeToUnsafeMetric(oracle_rmttu=oracle_metrics['rmttu']),
    TotalBudgetUsed(),
    ObservedJailbreaks(),
    CostPerJailbreakMetric(),
    SurvivalQuantilesMetric(oracle_quantiles=oracle_metrics['quantiles'], quantiles=[0.25, 0.50, 0.75])
])

# Evaluate metrics
metrics_results = engine.evaluate(traj_data)

# Display results in a DataFrame for readability
results_df = pd.DataFrame([metrics_results]).T
results_df.columns = ["Value"]
display(results_df)

## 7) Notes for running on real datasets
- Replace `is_real=False` with `is_real=True` in `get_data(...)`.
- Set `dataset_name` and `data_setup` to one of your prepared real-data folders.
- Tune `budget_per_sample`, `tau_prior`, `projection`, and `n1` to compare metric estimation efficiency across allocators.